In [51]:
import os, sys, subprocess, importlib, shutil, time
import numpy as np

# --- Project setup (works on Colab and locally) ---
_PROJECT_ROOT = None
for _candidate in [os.getcwd(), "/content/Distance-Estimator-NN", "/home/adsarver/Distance-Estimator-NN"]:
    if os.path.isfile(os.path.join(_candidate, "data.py")):
        _PROJECT_ROOT = _candidate
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing data.py")

os.chdir(_PROJECT_ROOT)
sys.path.insert(0, _PROJECT_ROOT)

# Pull dataset from Google Drive if not present
DATA_DIR = os.path.join(_PROJECT_ROOT, "rgbd-scenes-v2", "imgs")
if not os.path.isdir(DATA_DIR):
    # Mount Google Drive (Colab only)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass  # not on Colab
    
    DRIVE_ZIP = "/content/drive/MyDrive/rgbd-scenes-v2_imgs.zip"
    LOCAL_ZIP = os.path.join(_PROJECT_ROOT, "rgbd-scenes-v2_imgs.zip")
    if not os.path.isfile(DRIVE_ZIP):
        raise FileNotFoundError(f"Dataset zip not found on Google Drive at {DRIVE_ZIP}")
    print("Copying dataset from Google Drive ...")
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print("Unzipping ...")
    subprocess.check_call(["unzip", "-q", LOCAL_ZIP, "-d", _PROJECT_ROOT])
    os.remove(LOCAL_ZIP)
    print("Dataset ready.")

# Force-reload local modules in case they were cached
for _mod in ["data", "model"]:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

print(f"Working directory: {os.getcwd()}")

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

from data import RGBDDataset, scene_collate_fn
from model import DistanceNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
IMG_SIZE = 640
HIDDEN_SIZE = 128
LSTM_LAYERS = 1
MEMORY_LENGTH = 60
MEMORY_STRIDE = 2
LR = 1e-3
EPOCHS = 100
TBTT_LEN = 64          # truncation window: backprop every N frames
VAL_SPLIT = 0.25        # fraction of scenes held out for validation
PRINT_EVERY = 5       # print frame progress every N frames

# --- Scene train/val split ---
from glob import glob
from collections import defaultdict

all_files = sorted(glob(os.path.join(DATA_DIR, "*/*-color.png")))
_scenes_map = defaultdict(list)
for f in all_files:
    _scenes_map[os.path.basename(os.path.dirname(f))].append(f)

all_scene_names = sorted(_scenes_map.keys())
np.random.seed(42)
np.random.shuffle(all_scene_names)

n_val = max(1, int(len(all_scene_names) * VAL_SPLIT))
val_scene_names = all_scene_names[:n_val]
train_scene_names = all_scene_names[n_val:]

# --- DataLoaders (one scene per batch) ---
train_ds = RGBDDataset(DATA_DIR, scene_names=train_scene_names)
val_ds   = RGBDDataset(DATA_DIR, scene_names=val_scene_names, random_seed=123)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=scene_collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=scene_collate_fn, num_workers=0)

print(f"Scenes — train: {len(train_ds)}, val: {len(val_ds)}")
print(f"  Train scenes: {train_scene_names}")
print(f"  Val scenes:   {val_scene_names}")

Working directory: /home/adsarver/Distance-Estimator-NN
Using device: cuda
Scenes — train: 11, val: 3
  Train scenes: ['scene_13', 'scene_06', 'scene_09', 'scene_03', 'scene_02', 'scene_14', 'scene_05', 'scene_08', 'scene_11', 'scene_04', 'scene_07']
  Val scenes:   ['scene_10', 'scene_12', 'scene_01']


In [52]:
# --- Model ---
model = DistanceNN(
    hidden_size=HIDDEN_SIZE,
    lstm_num_layers=LSTM_LAYERS,
    memory_length=MEMORY_LENGTH,
    memory_stride=MEMORY_STRIDE,
    img_size=IMG_SIZE,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler("cuda")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 1,282,169


In [53]:
def safe_detach_head(head):
    """Detach hidden state & buffer only if they've been initialized."""
    if head.hx is not None:
        head.hx = tuple(h.detach() for h in head.hx)
    if head.buffer is not None:
        head.buffer = head.buffer.detach()


def masked_smooth_l1(pred, target):
    """Smooth L1 loss only on pixels where ground-truth depth > 0 (valid)."""
    valid = target > 0
    if not valid.any():
        return (pred * 0).sum()  # keep in graph, contributes zero
    return F.smooth_l1_loss(pred[valid], target[valid])


def run_epoch(loader, model, optimizer, scaler, device, is_train=True):
    """
    TBTT epoch with mixed-precision, print-based progress, and validation metrics.
    Frames are loaded lazily one at a time to avoid OOM.
    Returns (avg_loss, metrics_dict | None).
    """
    mode = "Train" if is_train else "Val"
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_frames = 0
    dataset = loader.dataset
    n_scenes = len(dataset)

    # Validation metric accumulators
    sum_mae = 0.0
    sum_sq_err = 0.0
    sum_delta1 = 0
    total_val_pixels = 0

    epoch_t0 = time.time()

    for si, scene_meta in enumerate(loader):
        T = scene_meta["T"]

        if T < 2:
            print(f"  Scene {si+1}/{n_scenes} ({scene_meta['scene']}): too few frames, skipping.")
            continue

        model.reset_lstm()
        chunk_loss = torch.tensor(0.0, device=device)
        chunk_frames = 0
        scene_t0 = time.time()

        for t in range(T):
            frame = dataset.load_frame(scene_meta, t)
            crop_h, crop_w = frame["crop_dim"]
            if crop_h < 2 or crop_w < 2:
                continue

            img = F.interpolate(frame["rgb"].unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                                mode="bilinear", align_corners=False).to(device)
            bbox_t = frame["bbox"].unsqueeze(0).to(device)
            obj_img = frame["rgb_crop"].unsqueeze(0).to(device)
            gt_depth = frame["depth_crop"].squeeze(0).to(device)

            gt_max = gt_depth.max()
            gt_norm = gt_depth / gt_max if gt_max > 0 else gt_depth

            if is_train:
                with autocast("cuda"):
                    pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                    loss = masked_smooth_l1(pred, gt_norm)
                chunk_loss = chunk_loss + loss
            else:
                with torch.no_grad(), autocast("cuda"):
                    pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                    chunk_loss = chunk_loss + masked_smooth_l1(pred, gt_norm)

                    # --- Per-pixel validation metrics ---
                    valid = gt_norm > 0
                    if valid.any():
                        p = pred[valid].float()
                        g = gt_norm[valid].float()
                        n_px = p.numel()

                        sum_mae += (p - g).abs().sum().item()
                        sum_sq_err += ((p - g) ** 2).sum().item()
                        ratio = torch.max(p / g.clamp(min=1e-6), g / p.clamp(min=1e-6))
                        sum_delta1 += (ratio < 1.25).sum().item()
                        total_val_pixels += n_px

            del img, bbox_t, obj_img, gt_depth, gt_norm, pred, frame
            chunk_frames += 1
            total_frames += 1

            # --- TBTT boundary ---
            if is_train and chunk_frames % TBTT_LEN == 0:
                avg_loss = chunk_loss / TBTT_LEN
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(avg_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

                total_loss += chunk_loss.item()
                chunk_loss = torch.tensor(0.0, device=device)

                safe_detach_head(model.ctx_head)
                safe_detach_head(model.shape_head)
                safe_detach_head(model.obj_head)

            # Periodic frame progress (include current chunk in display)
            if (t + 1) % PRINT_EVERY == 0:
                running = (total_loss + chunk_loss.item()) / max(total_frames, 1)
                print(f"    [{((t+1) / T) * 100:.1f}%] Frame {t+1}/{T}  avg_loss={running:.5f}", end='\r')

        # Flush remaining chunk (train: backprop leftover; val: always accumulate)
        leftover = chunk_frames % TBTT_LEN
        if is_train and leftover > 0 and chunk_loss.requires_grad:
            avg_loss = chunk_loss / leftover
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(avg_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += chunk_loss.item()

        del chunk_loss
        torch.cuda.empty_cache()

        scene_dt = time.time() - scene_t0
        print(f"  [{mode}] Scene {si+1}/{n_scenes} ({scene_meta['scene']}) "
              f"| {T} frames in {scene_dt:.1f}s "
              f"| avg_loss={total_loss / max(total_frames, 1):.5f}")

    epoch_dt = time.time() - epoch_t0
    avg_loss = total_loss / max(total_frames, 1)
    print(f"  {mode} done — {total_frames} frames in {epoch_dt:.1f}s | avg_loss={avg_loss:.6f}")

    if not is_train and total_val_pixels > 0:
        metrics = {
            "mae":    sum_mae / total_val_pixels,
            "rmse":   (sum_sq_err / total_val_pixels) ** 0.5,
            "delta1": sum_delta1 / total_val_pixels,
        }
        return avg_loss, metrics

    return avg_loss, None

In [54]:
# --- Training Loop (TBTT, scene-level streaming via DataLoader) ---
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "mae": [], "rmse": [], "delta1": [], "lr": []}

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}\nEpoch {epoch}/{EPOCHS}\n{'='*50}")
    
    train_loss, _ = run_epoch(train_loader, model, optimizer, scaler, device, is_train=True)
    print()
    val_loss, val_metrics = run_epoch(val_loader, model, optimizer, scaler, device, is_train=False)
    print()

    scheduler.step()
    cur_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(cur_lr)
    if val_metrics:
        history["mae"].append(val_metrics["mae"])
        history["rmse"].append(val_metrics["rmse"])
        history["delta1"].append(val_metrics["delta1"])

    tag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        tag = " * saved"

    print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {cur_lr:.2e}{tag}")
    if val_metrics:
        print(f"  Val MAE: {val_metrics['mae']:.5f} | "
              f"RMSE: {val_metrics['rmse']:.5f} | "
              f"δ<1.25: {val_metrics['delta1']:.2%}")


Epoch 1/100


  [Train] Scene 1/11 (scene_14) | 659 frames in 29.8s | avg_loss=0.02573
  [Train] Scene 2/11 (scene_09) | 732 frames in 28.5s | avg_loss=0.02753
  [Train] Scene 3/11 (scene_04) | 868 frames in 31.4s | avg_loss=0.02343
  [Train] Scene 4/11 (scene_06) | 1048 frames in 38.6s | avg_loss=0.02839
  [Train] Scene 5/11 (scene_03) | 861 frames in 28.8s | avg_loss=0.02732
  [Train] Scene 6/11 (scene_11) | 640 frames in 22.5s | avg_loss=0.02802
  [Train] Scene 7/11 (scene_13) | 462 frames in 18.5s | avg_loss=0.02765
  [Train] Scene 8/11 (scene_08) | 925 frames in 29.9s | avg_loss=0.02771
  [Train] Scene 9/11 (scene_07) | 943 frames in 35.7s | avg_loss=0.02777
  [Train] Scene 10/11 (scene_02) | 834 frames in 34.1s | avg_loss=0.02788
  [Train] Scene 11/11 (scene_05) | 1128 frames in 43.7s | avg_loss=0.02955
  Train done — 9100 frames in 341.5s | avg_loss=0.029549

  [Val] Scene 1/3 (scene_10) | 716 frames in 15.5s | avg_loss=0.02370
  [Val] Scene 2/3 (scene_12) | 723 frames in 15.7s | avg_loss=0.0

KeyboardInterrupt: 